# Homework 13 — Productization

This homework is self-contained: I train a tiny model, save it with `joblib`, put it behind a Flask API
with two routes, call that API from this notebook, and document how to run it.

*(Note: the server runs on port **5001**, not 5000 — on macOS, port 5000 is often taken by AirPlay
Receiver.)*

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install scikit-learn
# !pip install joblib
# !pip install flask
# !pip install requests

## 1. Generate data and train a model

Run as-is. `os.makedirs('model', exist_ok=True)` must come **before** `joblib.dump`, or the save fails
with `FileNotFoundError` because `model/` doesn't exist yet.

In [2]:
import os
import joblib
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

# the dataset for this homework - generated, not loaded
X, y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)

model = LinearRegression()
model.fit(X, y)

os.makedirs('model', exist_ok=True)          # BEFORE the dump, not after
joblib.dump(model, 'model/model.pkl')

# prove the file on disk is usable: load it back and predict with the loaded copy
reloaded = joblib.load('model/model.pkl')
print('saved to model/model.pkl')
print('prediction from the reloaded model:', float(reloaded.predict([[0.1, 0.2]])[0]))

saved to model/model.pkl
prediction from the reloaded model: 23.58961171297328


## 2. Write `app.py`

The model load stays **at the top of the file** — it runs once when the app starts, not inside a route.
A route that loads the model on every request re-reads the file from disk for every caller.

In [3]:
app_code = '''from flask import Flask, request, jsonify
import joblib

# Loaded ONCE at startup — not inside a route.
# (A route that calls joblib.load on every request would re-read the file from
#  disk for every single caller, which is the mistake this task teaches us to avoid.)
model = joblib.load('model/model.pkl')
app = Flask(__name__)


@app.route('/predict', methods=['POST'])
def predict_post():
    data = request.get_json(silent=True) or {}
    features = data.get('features')

    if not isinstance(features, list) or len(features) != 2:
        return jsonify({'error': 'expected {"features": [f1, f2]} with exactly 2 numbers'}), 400

    try:
        pred = float(model.predict([features])[0])
    except (ValueError, TypeError):
        return jsonify({'error': 'features must be numeric'}), 400

    return jsonify({'prediction': pred})


@app.route('/predict/<f1>/<f2>', methods=['GET'])
def predict_get(f1, f2):
    # f1 and f2 arrive as STRINGS; convert to float and reject non-numbers.
    try:
        f1f, f2f = float(f1), float(f2)
    except ValueError:
        return jsonify({'error': 'path parameters must be numbers'}), 400

    pred = float(model.predict([[f1f, f2f]])[0])
    return jsonify({'prediction': pred})


if __name__ == '__main__':
    # Port 5001 (not 5000): on macOS, port 5000 is often taken by AirPlay Receiver.
    app.run(host='127.0.0.1', port=5001)
'''

with open('app.py', 'w') as f:
    f.write(app_code)
print('wrote app.py')

wrote app.py


## 3. Launch the server

This starts Flask in the background (the same thing as `python app.py` in a separate terminal) and waits
until the server is reachable before moving on.

In [4]:
import subprocess, sys, time, socket

subprocess.Popen([sys.executable, 'app.py'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    time.sleep(0.5)
    s = socket.socket(); s.settimeout(0.3)
    try:
        s.connect(('127.0.0.1', 5001)); s.close(); break
    except OSError:
        s.close()
print('Flask server is up at http://127.0.0.1:5001')

Flask server is up at http://127.0.0.1:5001


## 4. Call your own API

Two good calls (POST and GET) and two deliberately bad calls — each must return JSON and an HTTP 400 for
the bad ones. This output is the testing evidence.

In [5]:
import requests

BASE = 'http://127.0.0.1:5001'

r1 = requests.post(BASE + '/predict', json={'features': [0.1, 0.2]}, timeout=5)
print('POST /predict (good)          ', r1.status_code, r1.text.strip())

r2 = requests.get(BASE + '/predict/0.1/0.2', timeout=5)
print('GET  /predict/0.1/0.2         ', r2.status_code, r2.text.strip())

r3 = requests.get(BASE + '/predict/abc/0.2', timeout=5)
print('GET  /predict/abc/0.2 (bad)   ', r3.status_code, r3.text.strip())

r4 = requests.post(BASE + '/predict', json={'wrong_key': [0.1, 0.2]}, timeout=5)
print('POST /predict (missing key)   ', r4.status_code, r4.text.strip())

POST /predict (good)           200 {"prediction":23.58961171297328}
GET  /predict/0.1/0.2          200 {"prediction":23.58961171297328}
GET  /predict/abc/0.2 (bad)    400 {"error":"path parameters must be numbers"}
POST /predict (missing key)    400 {"error":"expected {\"features\": [f1, f2]} with exactly 2 numbers"}


## 5. `README.md`

`README.md` lives next to `app.py` and documents how to start the server and call both routes. It is
written separately (with the real responses pasted in) after this notebook runs.